# 实践 3.2  从基因型矩阵构建G矩阵并完成GBLUP

《统计遗传与智能育种学》第三章 · 电子资源 `03_mixed_model_gblup.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：03_mixed_model_gblup.ipynb。实践以固定随机种子模拟基因型和数量性状，利用模拟时已知的等位基因频率构建基因组关系矩阵，通过一维REML剖面似然估计残差方差与遗传方差之比，并在训练—测试划分下完成GBLUP预测。实践中同时报告基因组估计育种值（genomic estimated breeding value, GEBV）与模拟真实育种值的相关。输出包括估计遗传力、G矩阵摘要、表型预测相关以及GEBV准确度。

模型：y = Xβ + u + e  
u ~ N(0, Gσ²g),  e ~ N(0, Iσ²e)  
验收：Notebook从头运行，无隐藏状态，固定随机种子后输出可重复。

## 输入、完整代码、实际输出与解释

输入与可复现条件：260个模拟个体的基因型与数量性状；SEED=302；按已知模拟频率构建全体G矩阵，在训练集上用一维REML剖面估计方差比。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=302
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 302


In [2]:
from scipy.linalg import eigh
from scipy.optimize import minimize_scalar

n,m=260,420; p=rng.uniform(.08,.5,m); X=rng.binomial(2,p,(n,m)).astype(float); b=rng.normal(0,.08,m); tbv=X@b; y=tbv+rng.normal(0,1.4,n)
W=X-2*p; G=W@W.T/(2*np.sum(p*(1-p))); print('G shape',G.shape,'diag mean',round(np.diag(G).mean(),3),'offdiag mean',round((G.sum()-np.trace(G))/(n*(n-1)),4))

# 一维REML剖面：V = sigma_g^2 G + sigma_e^2 I，只优化lambda=sigma_e^2/sigma_g^2
tr=np.arange(200); te=np.arange(200,n); K=G[np.ix_(tr,tr)]; yt=y[tr]; one=np.ones(len(tr))
def reml_nll(loglam):
 lam=np.exp(loglam); V=K+lam*np.eye(len(tr)); L=np.linalg.cholesky(V); Vi_y=np.linalg.solve(L.T,np.linalg.solve(L,yt)); Vi_1=np.linalg.solve(L.T,np.linalg.solve(L,one)); mu=(one@Vi_y)/(one@Vi_1); r=yt-mu; q=r@np.linalg.solve(L.T,np.linalg.solve(L,r)); sg2=q/(len(tr)-1); return (len(tr)-1)*np.log(sg2)+2*np.log(np.diag(L)).sum()+np.log(one@Vi_1)
opt=minimize_scalar(reml_nll,bounds=(-5,5),method='bounded'); lam=float(np.exp(opt.x)); h2=1/(1+lam)
V=K+lam*np.eye(len(tr)); Vi_1=np.linalg.solve(V,one); Vi_y=np.linalg.solve(V,yt); mu=(one@Vi_y)/(one@Vi_1)
a=np.linalg.solve(V,yt-mu); pred=mu+G[np.ix_(te,tr)]@a
print('lambda',round(lam,3),'estimated h2',round(h2,3),'test phenotype r',round(np.corrcoef(pred,y[te])[0,1],3),'GEBV accuracy',round(np.corrcoef(pred,tbv[te])[0,1],3))


G shape (260, 260) diag mean 1.005 offdiag mean 0.0002
lambda 0.452 estimated h2 0.689 test phenotype r 0.259 GEBV accuracy 0.234


结果解释：G矩阵平均对角约1.005、平均非对角约0，符合本模拟的中心化尺度；在当前G的缩放方式和仅含加性遗传方差、残差方差两个组分的模型下，由λ=σ²e/σ²g得到估计h²=1/(1+λ)≈0.689。若改变G的尺度或加入其他随机效应，这一换算需要重新推导。留出测试集中表型相关0.259、GEBV与模拟真实育种值相关0.234，强调预测评价必须明确参照对象，不能把训练拟合或表型相关直接称为育种值准确度。

可修改练习：把训练个体从200改为130和221，重复多个SEED，分别汇总表型相关和GEBV准确度的分布。